# Criando um Assistente de Investimentos com RPA e IA Generativa
## Desafio Final DIO.me

Este notebook executa a etapa de **RPA (Robotic Process Automation)** do projeto:
1. Conecta-se à página de simulação de clientes hospedada no GitHub Pages.
2. Extrai os dados cadastrais (Nome, E-mail, Saldo e Perfil de Investidor) usando **BeautifulSoup**.
3. Envia os dados estruturados via requisição `POST` para o **Webhook do N8N**.

### 1. Instalação das Dependências

In [ ]:
!pip install requests beautifulsoup4

### 2. Extração via RPA e Envio ao N8N Webhook

> 💡 **Configuração do Webhook:**
> Substitua a variável `N8N_WEBHOOK` com a URL fornecida pelo seu nó Webhook no N8N (Cloud ou Local).
> Exemplo Local: `http://localhost:5678/webhook/clientes`
> Exemplo Cloud: `https://seu-usuario.app.n8n.cloud/webhook/clientes`

In [ ]:
import requests
import json
from bs4 import BeautifulSoup

# 1. URL da página pública de clientes
URL_PAGINA = "https://digitalinnovationone.github.io/dio-lab-assistente-investimentos-rpa-n8n/"

# 2. URL do Webhook do seu N8N (Atualize com o seu endpoint)
N8N_WEBHOOK = "http://localhost:5678/webhook/clientes"

print("🌐 [1/3] Acessando a página para extração RPA...")
resposta = requests.get(URL_PAGINA)
resposta.raise_for_status()

# 3. Parse do HTML com BeautifulSoup
soup = BeautifulSoup(resposta.text, "html.parser")
linhas = soup.select("#clientes tbody tr")

clientes = []
for linha in linhas:
    colunas = linha.find_all("td")
    if len(colunas) >= 4:
        cliente = {
            "nome": colunas[0].get_text(strip=True),
            "email": colunas[1].get_text(strip=True),
            "saldo": colunas[2].get_text(strip=True),
            "perfil": colunas[3].get_text(strip=True),
        }
        clientes.append(cliente)

print(f"✅ [2/3] {len(clientes)} clientes extraídos com sucesso:")
for c in clientes:
    print(f"  - {c['nome']:<18} | Perfil: {c['perfil']:<11} | Saldo: {c['saldo']}")

# 4. Disparo para o Webhook do N8N
print(f"\n🚀 [3/3] Enviando lote para o Webhook do N8N: {N8N_WEBHOOK}")
try:
    resp_webhook = requests.post(N8N_WEBHOOK, json={"clientes": clientes}, timeout=30)
    print(f"📥 Código de Retorno HTTP: {resp_webhook.status_code}")
    
    if resp_webhook.status_code == 200:
        print("🎉 Sucesso! Retorno do Workflow N8N:")
        print(json.dumps(resp_webhook.json(), indent=2, ensure_ascii=False))
    else:
        print(f"⚠️ Webhook retornou status {resp_webhook.status_code}:")
        print(resp_webhook.text)
except requests.exceptions.ConnectionError:
    print("❌ Erro de conexão: O N8N não está respondendo nesta URL.")
    print("Verifique se o workflow está ativo ou clique em 'Listen for test event' no N8N.")
